In [1]:
import os, re, tempfile
from pathlib import Path
import subprocess as sp
import pandas as pd
from math import dist
from tqdm.notebook import tqdm
from typing import List, Tuple, Optional

from schrodinger import structure as sstruct
from schrodinger.structutils import analyze

SCHRO = "/opt/schrodinger2025-2"

In [2]:
outdir = Path("undetected_sitemap")
outdir.mkdir(exist_ok=True)

In [3]:
siteresd = pd.read_pickle("sitemap_undetected_siteres.pkl")
siteresd

{'7gqu': {'7gqu': [570,
   571,
   572,
   703,
   704,
   705,
   706,
   707,
   711,
   725,
   726,
   727,
   728,
   729,
   844,
   845,
   846,
   847,
   849,
   850,
   891,
   895,
   898,
   913,
   916,
   917,
   918,
   919,
   920],
  '7gqt': [570,
   571,
   572,
   703,
   704,
   705,
   706,
   707,
   711,
   725,
   726,
   727,
   728,
   729,
   844,
   845,
   846,
   847,
   849,
   850,
   891,
   895,
   898,
   913,
   916,
   917,
   918,
   919,
   920],
  '8yle': [53,
   54,
   55,
   186,
   187,
   188,
   189,
   190,
   194,
   208,
   209,
   210,
   211,
   212,
   327,
   328,
   329,
   330,
   332,
   333,
   374,
   378,
   381,
   396,
   399,
   400,
   401,
   402,
   403],
  '6yhr': [570,
   571,
   572,
   703,
   704,
   705,
   706,
   707,
   711,
   725,
   726,
   727,
   728,
   729,
   844,
   845,
   846,
   847,
   849,
   850,
   891,
   895,
   898,
   913,
   916,
   917,
   918,
   919,
   920],
  '8pfp': [570,
   571,
   572,

In [ ]:
siteresd["8jp0"].pop("8sgi")

# Functions

## General

In [4]:
def run(cmd: List[str], cwd: Optional[str] = None, **kwargs) -> None:
    # print(f"[CMD] {cmd if 'shell' in kwargs else ' '.join(cmd)}")
    try:
        p = sp.run(cmd, check=True, cwd=cwd, text=True, **kwargs)
        # print(p.stdout, p.stderr)
    except sp.CalledProcessError as e:
        print("[CMD FAILED]", e.returncode, e.cmd)
        print("[STDOUT]\n", e.stdout)
        print("[STDERR]\n", e.stderr)
        raise
    return


def which(tool: str) -> str:
    """Return full path to a Schrödinger CLI tool."""
    p = Path(SCHRO) / tool
    if p.exists():
        return str(p)
    # Some tools live at the root (e.g., $SCHRODINGER/prepwizard), others under subdirs.
    for sub in ["", "utilities", "shape_screen", "glide", "sitemap", "ligprep", "epik", "vsw"]:
        pp = Path(SCHRO) / sub / tool
        if pp.exists():
            return str(pp)
    return tool  # fallback on PATH


def ensure_outdir(d: Path):
    d.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Output directory: {d}")

# Running

In [5]:
glidedir = outdir.resolve()

In [6]:
sites = []

for holo, holod in tqdm(siteresd.items()):
    for apo, siteres in tqdm(holod.items(), desc=holo):
        path = glidedir / holo / apo
        path.mkdir(exist_ok=True, parents=True)
        outf = path / f"{apo}_sitemap_out.maegz"
        
        if not outf.exists():
            prep = Path(f"glide/{holo}/preps/{apo}_prep.mae").resolve()
    
            st = next(sstruct.StructureReader(prep))
            asl = f"res.num {', '.join(map(str, siteres))}"
            ats = [st.atom[i] for i in analyze.evaluate_asl(st, asl)]
            cx, cy, cz = [sum(getattr(a, c) for a in ats)/len(ats) for c in ("x", "y", "z")]
    
            centermae = path / f"{apo}_sitecenter.mae"
            na_st = sstruct.create_new_structure()# Structure. create_new()   # empty structure
            a = na_st.addAtom("Na", cx, cy, cz)
            a.pdbres = "NA"; a.resnum = 999; a.pdbname = "NA"; a.chain = 'Z'
            na_st.write(centermae)
    
            sitemap_in = path / f"{apo}_sitemap.in"    
            sitemap_in.write_text(f"""
PROTEIN {prep}
LIGMAE {centermae}
SITEBOX 8
MODPHOBIC 0
ENCLOSURE 0.4
MAXVDW 0.55
COMPACT_MODE_THRESHOLD 800
KEEPLOGS yes
            """)

            with open(path / "run.log", "w") as logf:
                run(
                    " ".join((
                        SCHRO + "/sitemap",
                        sitemap_in.name,
                        "-WAIT -HOST localhost:1 -TMPLAUNCHDIR"
                    )),
                    cwd=path, shell=True,
                    stdout=logf, stderr=logf
                )

        
        if outf.exists():
            sts = list(sstruct.StructureReader(outf))        # outf = sitemap .maegz
            na  = next(a for a in sts[-1].atom if a.element == 'Na')  # Na in last entry
            
            for i, site in enumerate(sts[:-2], 1):           # all site entries
                mind = min(dist((na.x, na.y, na.z), (s.x, s.y, s.z)) for s in site.atom)
                sites.append({
                    "pdb": holo,
                    "apo": apo,
                    "SiteMap": i,
                    "allositeCoG-SiteMap dist": mind,
                    "volume": site.property.get("r_sitemap_volume", "") 
                })
        else:
            sites.append({
                "pdb": holo,
                "apo": apo,
                "SiteMap": False,
                # "allositeCoG-SiteMap dist": None,
                # "volume": None
            })

sites

  0%|          | 0/8 [00:00<?, ?it/s]

7gqu:   0%|          | 0/6 [00:00<?, ?it/s]

7yg5:   0%|          | 0/4 [00:00<?, ?it/s]

8aq6:   0%|          | 0/9 [00:00<?, ?it/s]

8f4s:   0%|          | 0/70 [00:00<?, ?it/s]

8qni:   0%|          | 0/6 [00:00<?, ?it/s]

8v81:   0%|          | 0/19 [00:00<?, ?it/s]

9dnm:   0%|          | 0/16 [00:00<?, ?it/s]

8jp0:   0%|          | 0/5 [00:00<?, ?it/s]

OSError: File does not exist: /data/fnerin/ensembles/glide/8jp0/preps/8sgi_prep.mae

In [8]:
df = pd.DataFrame(sites)
df

,pdb,apo,SiteMap,allositeCoG-SiteMap dist,volume
0,7gqu,7gqu,1,0.000000,276.458
1,7gqu,7gqt,1,3.162278,471.282
2,7gqu,8yle,1,4.123106,462.364
3,7gqu,6yhr,1,5.477226,325.850
4,7gqu,6yhr,2,4.242641,133.084
...,...,...,...,...,...
139,9dnm,8go8,1,0.000000,15.092
140,8jp0,8jp0,1,0.000000,359.464
141,8jp0,8sgj,1,6.633250,66.885
142,8jp0,9iv8,1,7.000000,105.301


In [11]:
df.sort_values("volume").iloc[:50]

,pdb,apo,SiteMap,allositeCoG-SiteMap dist,volume
15,8aq6,7sns,1,3.162278,5.488
139,9dnm,8go8,1,0.000000,15.092
13,8aq6,7sny,1,2.449490,17.493
124,9dnm,9dnm,1,1.414214,18.865
19,8aq6,7snt,1,4.123106,19.551
12,8aq6,7snr,1,3.741657,20.923
16,8aq6,7vsx,1,3.000000,22.981
18,8aq6,5b0u,1,2.449490,23.324
11,8aq6,8aq6,1,3.741657,25.039
128,9dnm,8hst,1,2.236068,26.411


In [19]:
df.to_pickle("sitemap_undetected_results.pkl")

In [16]:
rows

2

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pdbs = df["pdb"].unique()
n = len(pdbs)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows), squeeze=False)

for ax, pdb in zip(axes.flatten(), pdbs):
    sub = df[df["pdb"] == pdb]

    # Boxplot
    sns.boxplot(data=sub, x="pdb", y="volume", ax=ax, color="lightgray")

    # Scatter of all points
    sns.stripplot(data=sub, x="pdb", y="volume", ax=ax,
                  color="black", alpha=0.6, size=4)

    # Highlight point where apo == pdb
    highlight = sub[sub["apo"] == pdb]
    ax.scatter([0]*len(highlight), highlight["volume"],
               color="red", s=60, zorder=5)

    ax.set_title(pdb)
    ax.set_xlabel("")
    ax.set_xticks([])

# # Hide unused axes
# for ax in axes.flatten()[n:]:
#     ax.axis("off")

plt.tight_layout()
plt.show()

In [14]:
print("hi")

hi
